# V1-11: 통합 후보 기반 기업 확정·재질문 정책 비교

10번에서 후보 생성 방식 B(exact + fuzzy 통합)를 다음 단계의 기준으로 두었다. 이번에는 B가 만든 후보를 바탕으로 언제 기업을 자동 확정하고, 언제 사용자에게 재질문해야 하는지 비교한다.

이 notebook은 LLM·LangChain을 사용하지 않는다. 기업 확정은 OpenDART 데이터와 결정 규칙으로 평가한다.

## 1. 비교할 정책

- **A. 최상위 후보 자동 선택**: 후보가 하나 이상이면 정렬상 첫 번째 법인을 자동 확정한다.
- **B. 고유 exact만 자동 확정**: 정규화 정확 일치 법인이 하나일 때만 자동 확정한다. 동명 법인, fuzzy 후보만 있는 입력은 재질문한다.

A는 편의성 기준선이고, B는 현재 안전 정책 후보이다. `삼성`처럼 OpenDART에 정확히 같은 법인명이 하나 있더라도 일반 사용자의 의도가 모호할 수 있는 사례를 포함해 B의 한계도 확인한다.

In [ ]:
import io
import os
import re
import sys
import time
import xml.etree.ElementTree as ET
import zipfile
from math import ceil
from pathlib import Path
from statistics import median

from rapidfuzz import fuzz, process


In [ ]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

api_key = os.environ.get('OPENDART_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')


In [ ]:
sys.path.insert(0, str(project_root / 'src'))

from dart.client import CORP_CODE_URL, _get_bytes


## 2. OpenDART 기업 목록과 통합 후보 생성기

10번의 B 방식처럼 exact 후보와 fuzzy 후보를 모두 모은다. 다만 후보 생성과 최종 확정은 분리한다. 즉 fuzzy 후보가 함께 있어도, 고유 exact 법인이 하나인지 여부를 별도로 확인한다.

In [ ]:
def load_corp_records(api_key):
    zip_bytes = _get_bytes(CORP_CODE_URL, {'crtfc_key': api_key})
    zip_stream = io.BytesIO(zip_bytes)
    if not zipfile.is_zipfile(zip_stream):
        error_text = zip_bytes.decode('utf-8', errors='replace')
        raise RuntimeError(f'기업코드 응답이 ZIP이 아닙니다: {error_text}')
    with zipfile.ZipFile(zip_stream) as archive:
        xml_bytes = archive.read(archive.namelist()[0])

    root = ET.fromstring(xml_bytes)
    return [
        {child.tag: child.text or '' for child in item}
        for item in root.findall('list')
    ]

corp_records = load_corp_records(api_key)
print(f'전체 법인 레코드 수: {len(corp_records):,}')


In [ ]:
def normalize_name(value):
    return re.sub(r'[^0-9A-Z가-힣]', '', value.upper())

def build_name_index(records):
    index = {}
    for record in records:
        for name in {record['corp_name'], record['corp_eng_name']}:
            key = normalize_name(name)
            if key:
                index.setdefault(key, {})[record['corp_code']] = record
    return {key: list(by_code.values()) for key, by_code in index.items()}

name_index = build_name_index(corp_records)
name_keys = list(name_index)
print(f'정규화 이름 키 수: {len(name_keys):,}')


In [ ]:
FUZZY_KEY_LIMIT = 50
FUZZY_RESULT_LIMIT = 5

def exact_records(query):
    return name_index.get(normalize_name(query), [])

def union_candidates(query):
    candidates = {}
    for record in exact_records(query):
        candidates[record['corp_code']] = {'record': record, 'sources': {'exact'}, 'score': None}

    scored_keys = process.extract(
        normalize_name(query), name_keys, scorer=fuzz.ratio, limit=FUZZY_KEY_LIMIT
    )
    best_by_code = {}
    for key, score, _ in scored_keys:
        for record in name_index[key]:
            current = best_by_code.get(record['corp_code'])
            if current is None or score > current[1]:
                best_by_code[record['corp_code']] = (record, score)

    ranked_fuzzy = sorted(
        best_by_code.values(),
        key=lambda item: (item[1], bool(item[0]['stock_code'].strip()), item[0]['modify_date']),
        reverse=True,
    )[:FUZZY_RESULT_LIMIT]
    for record, score in ranked_fuzzy:
        current = candidates.setdefault(
            record['corp_code'],
            {'record': record, 'sources': set(), 'score': None},
        )
        current['sources'].add('fuzzy')
        current['score'] = max(current['score'] or 0, score)
    return candidates

def top_candidate(candidates):
    if not candidates:
        return None
    return sorted(
        candidates.values(),
        key=lambda candidate: (
            candidate['score'] or 0,
            bool(candidate['record']['stock_code'].strip()),
            candidate['record']['modify_date'],
        ),
        reverse=True,
    )[0]['record']


## 3. 평가 사례와 기대 상태

정식 기업명처럼 자동 확정이 가능한 입력과, 동명·오타·약칭·포괄 표현으로 재질문이 정답인 입력을 분리한다. `삼성`을 포괄 표현으로 보는 것은 OpenDART 사실이 아니라, 일반 사용자에게 안전한 동작을 우선하는 V1 제품 정책 정답이다. 이번 정책은 별칭 사전을 아직 포함하지 않으므로 `하닉`·`엘지` 등의 후보 품질 자체는 10번 결과로 판단하고, 여기서는 임의 자동 선택을 막는지만 본다.

In [ ]:
TEST_CASES = [
    {'질문 기업명': '삼성전자', '유형': '정식 명칭', '기대 상태': 'RESOLVED', '기대 종목코드': '005930'},
    {'질문 기업명': '삼성 전자', '유형': '띄어쓰기', '기대 상태': 'RESOLVED', '기대 종목코드': '005930'},
    {'질문 기업명': 'SK하이닉스', '유형': '정식 명칭', '기대 상태': 'RESOLVED', '기대 종목코드': '000660'},
    {'질문 기업명': 'sk 하이닉스', '유형': '영문 대소문자·띄어쓰기', '기대 상태': 'RESOLVED', '기대 종목코드': '000660'},
    {'질문 기업명': 'LG전자', '유형': '정식 명칭', '기대 상태': 'RESOLVED', '기대 종목코드': '066570'},
    {'질문 기업명': 'LG 전자', '유형': '띄어쓰기', '기대 상태': 'RESOLVED', '기대 종목코드': '066570'},
    {'질문 기업명': '현대자동차', '유형': '정식 명칭', '기대 상태': 'RESOLVED', '기대 종목코드': '005380'},
    {'질문 기업명': 'naver', '유형': '영문 소문자', '기대 상태': 'RESOLVED', '기대 종목코드': '035420'},
    {'질문 기업명': '삼전', '유형': '동명 법인·시장 약칭', '기대 상태': 'NEEDS_CONFIRMATION', '기대 종목코드': None},
    {'질문 기업명': '삼성전쟈', '유형': '한 글자 오타', '기대 상태': 'NEEDS_CONFIRMATION', '기대 종목코드': None},
    {'질문 기업명': '하이닉스', '유형': '부분 표현', '기대 상태': 'NEEDS_CONFIRMATION', '기대 종목코드': None},
    {'질문 기업명': '하닉', '유형': '시장 약칭', '기대 상태': 'NEEDS_CONFIRMATION', '기대 종목코드': None},
    {'질문 기업명': '엘지', '유형': '포괄 표현', '기대 상태': 'NEEDS_CONFIRMATION', '기대 종목코드': None},
    {'질문 기업명': '현대', '유형': '포괄 표현', '기대 상태': 'NEEDS_CONFIRMATION', '기대 종목코드': None},
    {'질문 기업명': '삼성', '유형': '정확 일치 법인이 있지만 포괄 표현', '기대 상태': 'NEEDS_CONFIRMATION', '기대 종목코드': None},
]

print(f'자동 확정 기대 사례: {sum(case["기대 상태"] == "RESOLVED" for case in TEST_CASES)}개')
print(f'재질문 기대 사례: {sum(case["기대 상태"] == "NEEDS_CONFIRMATION" for case in TEST_CASES)}개')


## 4. 정책 실행과 지표 계산

- **Auto-eligible Resolution Accuracy**: 자동 확정이 기대된 사례 중 올바른 종목코드로 자동 확정한 비율
- **Unsafe Selection Rate**: 자동 확정한 사례 중, 재질문이 정답이거나 종목코드가 틀린 비율
- **Abstention Accuracy**: 재질문이 정답인 사례를 실제로 재질문한 비율
- **Clarification Rate**: 전체 입력 중 재질문 비율


In [ ]:
def top_one_policy(query, candidates):
    selected = top_candidate(candidates)
    if selected is None:
        return 'NOT_FOUND', None
    return 'RESOLVED', selected

def unique_exact_policy(query, candidates):
    exact = exact_records(query)
    if len(exact) == 1:
        return 'RESOLVED', exact[0]
    if candidates:
        return 'NEEDS_CONFIRMATION', None
    return 'NOT_FOUND', None

POLICIES = {
    'A. 최상위 후보 자동 선택': top_one_policy,
    'B. 고유 exact만 자동 확정': unique_exact_policy,
}

WARMUP_ITERATIONS = 2
MEASURE_ITERATIONS = 10

def candidate_view(candidate):
    record = candidate['record']
    return {
        '법인명': record['corp_name'],
        'corp_code': record['corp_code'],
        '종목코드': record['stock_code'].strip() or None,
        '출처': '+'.join(sorted(candidate['sources'])),
        '문자열 유사 점수': round(candidate['score'], 1) if candidate['score'] is not None else None,
    }

def evaluate_policy(policy_name, policy):
    rows = []
    latencies = []
    for case in TEST_CASES:
        for _ in range(WARMUP_ITERATIONS):
            candidates = union_candidates(case['질문 기업명'])
            policy(case['질문 기업명'], candidates)

        measurements = []
        for _ in range(MEASURE_ITERATIONS):
            started_at = time.perf_counter()
            candidates = union_candidates(case['질문 기업명'])
            status, selected = policy(case['질문 기업명'], candidates)
            measurements.append((time.perf_counter() - started_at) * 1_000)

        views = sorted(
            (candidate_view(candidate) for candidate in candidates.values()),
            key=lambda row: (-(row['문자열 유사 점수'] or 0), row['법인명']),
        )
        rows.append({
            **case,
            '정책': policy_name,
            '실제 상태': status,
            '선택 법인명': selected['corp_name'] if selected else None,
            '선택 종목코드': selected['stock_code'].strip() if selected else None,
            '후보 수': len(candidates),
            '후보': views,
        })
        latencies.extend(measurements)

    auto_expected = [row for row in rows if row['기대 상태'] == 'RESOLVED']
    clarification_expected = [row for row in rows if row['기대 상태'] == 'NEEDS_CONFIRMATION']
    auto_selected = [row for row in rows if row['실제 상태'] == 'RESOLVED']
    summary = {
        '정책': policy_name,
        'Auto-eligible Resolution Accuracy': sum(
            row['실제 상태'] == 'RESOLVED' and row['선택 종목코드'] == row['기대 종목코드']
            for row in auto_expected
        ) / len(auto_expected),
        'Unsafe Selection Rate': sum(
            row['기대 상태'] != 'RESOLVED' or row['선택 종목코드'] != row['기대 종목코드']
            for row in auto_selected
        ) / len(auto_selected),
        'Abstention Accuracy': sum(
            row['실제 상태'] == 'NEEDS_CONFIRMATION'
            for row in clarification_expected
        ) / len(clarification_expected),
        'Clarification Rate': sum(row['실제 상태'] == 'NEEDS_CONFIRMATION' for row in rows) / len(rows),
        'p50 지연시간(ms)': median(latencies),
        'p95 지연시간(ms)': sorted(latencies)[ceil(len(latencies) * 0.95) - 1],
    }
    return rows, summary

all_rows = {}
summaries = []
for policy_name, policy in POLICIES.items():
    rows, summary = evaluate_policy(policy_name, policy)
    all_rows[policy_name] = rows
    summaries.append(summary)

for summary in summaries:
    print(summary)


## 5. 위험 사례 확인

아래 출력에서는 자동 선택이 발생한 경우를 직접 확인한다. 특히 `삼성`은 고유 exact가 하나여도 재질문이 기대되는 사례다. B가 이를 자동 확정하면 ‘고유 exact’ 규칙만으로는 충분하지 않다는 근거가 된다.

In [ ]:
FOCUS_INPUTS = ['삼성전자', '삼전', '삼성전쟈', '하닉', '엘지', '현대', '삼성']

for query in FOCUS_INPUTS:
    print(f'\n=== {query} ===')
    for policy_name, rows in all_rows.items():
        row = next(row for row in rows if row['질문 기업명'] == query)
        print({
            '정책': policy_name,
            '기대 상태': row['기대 상태'],
            '실제 상태': row['실제 상태'],
            '선택 법인명': row['선택 법인명'],
            '선택 종목코드': row['선택 종목코드'],
            '후보': row['후보'],
        })


## 6. 해석 기준

- A보다 B의 Unsafe Selection Rate가 낮고 Abstention Accuracy가 높으면, 자동 선택보다 재질문 정책이 안전하다는 근거가 된다.
- B가 `삼성`처럼 정확 일치 법인이 하나인 포괄 표현을 자동 확정하면, exact만으로는 사용자의 의도 모호성을 표현할 수 없다는 뜻이다. 이 경우 별칭·포괄 표현의 별도 보류 규칙을 다음 후보로 비교해야 한다.
- 이번 단계에서는 NOT_FOUND cutoff를 확정하지 않는다. 후보 생성과 재질문 정책을 먼저 정한 뒤 09번 cutoff를 B 기준으로 다시 평가한다.

이 notebook은 최종 pipeline이나 `src/` 코드를 변경하지 않는다.